# Data Training

## Our Training Methods

| Linear Models | Decision Trees | Support Vector Regressor | Evaluation |
|---|---|---|---|
| Standardized feature scaling, Polynomial feautres, Ridge/Lasso Regularization | Random Forest Regressor, Gradient Boosting, Hyperparameter tuning | SVR with kernel selection (RBF/Linear), C-Tuning, PCA dimensionality | RMSE & MAE |


### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### Organize Train and Test data

In [ ]:
train_data = pd.read_csv("../data/train.csv")
test_data = pd.read_csv("../data/test.csv")

TARGET = "log_retention"
ID_COLS = ["Adv_ID", "Genmodel_ID"]

FEATURES = [c for c in train.columns if c not in ID_COLS + [TARGET]]

NUM_COLS = ["age", "Runned_Miles", "Entry_price", "Engin_size", "Engine_power",
            "Annual_Tax", "Wheelbase", "Height", "Width", "Length",
            "Average_mpg", "Top_speed", "Seat_num", "Door_num"]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
groups = train["Genmodel_ID"]

#
CV = GroupKFold(n_splits=5)
SCORING = "neg_root_mean_squared_error"


### Evaluate Function

In [ ]:
def evaluate(model, name):
    pred_log = model.predict(X_test)
    rmse_log = np.sqrt(mean_squared_error(y_test, pred_log))
    mae_log = mean_absolute_error(y_test, pred_log)
    r2 = r2_score(y_test, pred_log)

    print(f"{name:18s} RMSE={rmse_log:.3f}  MAE={mae_log:.3f}  R²={r2:.3f}")
    return {"model": name, "RMSE": rmse_log, "MAE": mae_log, "R2": r2}

### Linear Model Code



In [ ]:
#Linear Pipeline 
def linear_pipeline(regressor):
    features = ColumnTransformer(
        [("poly", PolynomialFeatures(degree=2, include_bias=False), NUM_COLS)],
        remainder="passthrough",                # one-hot columns pass through untouched
    )
    return Pipeline([("features", features), ("model", regressor)])

#Run Ridge search: tune polynomial degree and alpha with grouped CV, refit best on all training data
ridge_grid = GridSearchCV(
    linear_pipeline(Ridge()),
    param_grid={
        "features__poly__degree": [1, 2], 
        "model__alpha": [0.1, 1, 10, 100, 1000],
    },
    cv=CV, scoring=SCORING, n_jobs=-1,
).fit(X_train, y_train, groups=groups)
print("Ridge best params:", ridge_grid.best_params_)

#Run Lasso search: tune polynomial degree and alpha with grouped CV, refit best on all training data 
lasso_grid = GridSearchCV(
    linear_pipeline(Lasso(max_iter=10000)),
    param_grid={
        "features__poly__degree": [1, 2],
        "model__alpha": [0.0001, 0.001, 0.01, 0.1],
    },
    cv=CV, scoring=SCORING, n_jobs=-1,
).fit(X_train, y_train, groups=groups)
print("Lasso best params:", lasso_grid.best_params_)

### Decision Trees Code

In [ ]:
#Random Forest: fixed settings, 300 trees, no tuning
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

#Run Gradient Boosting search: randomly try 30 hyperparameter combinations with grouped CV, refit best on all training data
gb_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_distributions={
        "n_estimators": [100, 200, 400],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [2, 3, 4, 5],
        "subsample": [0.7, 0.85, 1.0],
        "min_samples_leaf": [1, 3, 5],
    },
    n_iter=30, cv=CV, scoring=SCORING, random_state=42, n_jobs=-1,
).fit(X_train, y_train, groups=groups)
print("GB best params:", gb_search.best_params_)

#Describes which input columns the tuned gradient boosting model leaned on most when predicting retention 
importances = pd.Series(gb_search.best_estimator_.feature_importances_, index=FEATURES).nlargest(15)
print("\nTop GB feature importances:\n", importances.round(3).to_string(), "\n")

### SVR Code

In [ ]:
#SVR Pipeline
svr_pipe = Pipeline([("pca", PCA()), ("svr", SVR())])

#Check that PCA is beneficial to training
svr_params = []
for pca_option in [{"pca": [PCA()], "pca__n_components": [0.95, 0.99]},
                   {"pca": ["passthrough"]}]:
    svr_params.append({**pca_option, "svr__kernel": ["rbf"], "svr__C": [1, 10, 100],
                       "svr__gamma": ["scale", 0.01], "svr__epsilon": [0.05, 0.1]})
    svr_params.append({**pca_option, "svr__kernel": ["linear"], "svr__C": [0.1, 1, 10]})

#Run SVR Grid Search
svr_grid = GridSearchCV(
    svr_pipe, svr_params, cv=CV, scoring=SCORING, n_jobs=-1,
).fit(X_train, y_train, groups=groups)
print("SVR best params:", svr_grid.best_params_)

### Compare Models

In [ ]:
models = {
    "Baseline (mean)": baseline,
    "Ridge": ridge_grid,
    "Lasso": lasso_grid,
    "Random Forest": rf,
    "Gradient Boosting": gb_search,
    "SVR": svr_grid,
}

print("--- Test set results ---")
results = pd.DataFrame([evaluate(m, name) for name, m in models.items()])
print("\n" + results.sort_values("RMSE").to_string(index=False))

print("\n--- Best grouped-CV RMSE during tuning (log units) ---")
for name, s in [("Ridge", ridge_grid), ("Lasso", lasso_grid),
                ("Gradient Boosting", gb_search), ("SVR", svr_grid)]:
    print(f"{name:18s} {-s.best_score_:.3f}")

### Visual Representation of Comparing the Best Models

In [ ]:
best_name = results.sort_values("RMSE").iloc[0]["model"]
pred = models[best_name].predict(X_test)
residuals = y_test - pred

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(pred, residuals, alpha=0.3, s=10)
ax[0].axhline(0, color="red")
ax[0].set(xlabel="Predicted log_retention", ylabel="Residual",
          title=f"Residuals vs predicted ({best_name})")
ax[1].hist(residuals, bins=40)
ax[1].set(title="Residual distribution")
plt.tight_layout()
plt.savefig("residuals.png", dpi=120)
plt.show()